# 🍎 Health Calculator Agent Tutorial 🍏

Welcome to the **Health Calculator Agent** tutorial, where we'll showcase how to:
1. **Initialize** a project and use the Azure AI Foundry ecosystem
2. **Create an Agent** with **Code Interpreter** capabilities
3. **Perform BMI calculations** and **analyze nutritional data** with sample CSV files
4. **Generate** basic health insights and disclaimers

> #### Ensure you have completed the [`1-basics.ipynb`](./1-basics.ipynb) notebook before starting this one.

## Let's Dive In
We'll walk step-by-step, similar to our **Fun & Fit** sample, but with a focus on using **Code Interpreter** for numeric calculations and data analysis. Let's begin!

<img src="./seq-diagrams/2-code-interpreter.png" width="30%"/>




## 1. Initial Setup
We'll start by importing libraries, loading environment variables, and initializing an **AIProjectClient**. We'll also create a sample CSV for demonstration.


In [ ]:
# Import required libraries
import os
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient

# Load environment variables from the parent directory's .env
notebook_path = Path().absolute()
parent_dir = notebook_path.parent.parent
load_dotenv(parent_dir / '.env')

# Initialize the endpoint-based AIProjectClient and its OpenAI (Responses API) client
try:
    project_client = AIProjectClient(
        endpoint=os.environ["PROJECT_ENDPOINT"],
        credential=DefaultAzureCredential(),
    )
    openai_client = project_client.get_openai_client()
    print("✅ Successfully initialized AIProjectClient + OpenAI client")
except Exception as e:
    print(f"❌ Error initializing client: {str(e)}")

# Create sample CSV data for demonstration
def create_sample_data():
    try:
        data = {
            'Date': pd.date_range(start='2024-01-01', periods=7),
            'Calories': [2100, 1950, 2300, 2050, 1900, 2200, 2150],
            'Protein_g': [80, 75, 85, 78, 72, 82, 79],
            'Carbs_g': [250, 230, 270, 245, 225, 260, 255],
            'Fat_g': [70, 65, 75, 68, 63, 73, 71],
            'Fiber_g': [25, 22, 28, 24, 21, 26, 23]
        }
        df = pd.DataFrame(data)
        filename = "nutrition_data.csv"
        df.to_csv(filename, index=False)
        print(f"📄 Created sample data file: {filename}")
        return filename
    except Exception as e:
        print(f"❌ Error creating sample data: {e}")
        return None

sample_file = create_sample_data()

## 2. Create Health Calculator Agent 👩‍💻
We'll upload our sample CSV and then create an agent with **Code Interpreter** enabled. This agent can read the file, run Python code, and return results and visualizations.


In [ ]:
from azure.ai.projects.models import (
    PromptAgentDefinition,
    CodeInterpreterTool,
    AutoCodeInterpreterToolParam,
)

def create_health_calculator(file_path):
    """Create an agent with code interpreter for health/nutrition calculations."""
    try:
        # Upload the CSV via the OpenAI files API so the code interpreter can read it
        uploaded_file = openai_client.files.create(purpose="assistants", file=open(file_path, "rb"))
        print(f"✅ Uploaded CSV file, ID: {uploaded_file.id}")

        # Create the agent version with Code Interpreter enabled, attaching the uploaded file
        agent = project_client.agents.create_version(
            agent_name="health-calculator-agent",
            definition=PromptAgentDefinition(
                model=os.environ.get("MODEL_DEPLOYMENT_NAME", "gpt-5.4"),
                instructions="""
                You are a health calculator agent that can:
                1. Calculate and interpret BMI
                2. Analyze provided nutrition data
                3. Generate charts/plots and save them as files
                4. Include disclaimers that you are not a medical professional
                """,
                tools=[CodeInterpreterTool(container=AutoCodeInterpreterToolParam(file_ids=[uploaded_file.id]))],
            ),
            description="Health calculator agent with the code interpreter tool.",
        )
        print(f"🎉 Created health calculator agent '{agent.name}', version: {agent.version}")
        return agent, uploaded_file
    except Exception as e:
        print(f"❌ Error creating health calculator agent: {e}")
        return None, None

health_agent, uploaded_file = None, None
if sample_file:
    health_agent, uploaded_file = create_health_calculator(sample_file)

## 3. BMI Calculation with Code Interpreter
We'll open a conversation for BMI calculations. We'll feed in the user's height/weight, and ask the agent to show how it calculates BMI, interpret the result, and always disclaim professional advice.


In [ ]:
def calculate_bmi_with_agent(agent, height_inches, weight_pounds):
    """Calculate BMI using the code interpreter agent."""
    try:
        # Create a new conversation
        conversation = openai_client.conversations.create()
        print(f"📝 Created conversation for BMI calculation, ID: {conversation.id}")

        # Construct user message requesting BMI calculation
        user_text = (
            f"Calculate BMI for \n"
            f"Height: {height_inches} inches\n"
            f"Weight: {weight_pounds} pounds\n"
            "Please: \n"
            "1. Show calculation \n"
            "2. Interpret the result \n"
            "3. Include disclaimers \n"
        )

        # Run the agent on this input; the response is stored in the conversation
        response = openai_client.responses.create(
            conversation=conversation.id,
            input=user_text,
            extra_body={"agent_reference": {"name": agent.name, "type": "agent_reference"}},
        )
        print(f"🤖 BMI response status: {response.status}")
        return conversation, response
    except Exception as e:
        print(f"❌ Error during BMI calculation: {e}")
        return None, None

bmi_conversation, bmi_response = None, None
if health_agent:
    bmi_conversation, bmi_response = calculate_bmi_with_agent(health_agent, 70, 180)  # example: 5'10" and 180 lbs

## 4. Nutrition Analysis
We'll open another conversation where the user can ask the agent to analyze the **`nutrition_data.csv`** we uploaded. The agent can read the file, compute macros, produce a chart (saved as a PNG file), and disclaim that it's not offering personalized medical advice.


In [ ]:
def analyze_nutrition_data(agent):
    """Ask the agent to analyze the uploaded nutrition data."""
    try:
        conversation = openai_client.conversations.create()
        print(f"📝 Created conversation for nutrition analysis, ID: {conversation.id}")

        user_text = (
            "Analyze the CSV file with daily nutrition data.\n"
            "1. Compute average daily macros (calories, protein, carbs, fat, fiber).\n"
            "2. Use the code interpreter to plot the trends with matplotlib and "
            "SAVE the figure as a PNG file, then provide the file.\n"
            "3. Discuss any insights or disclaimers.\n"
        )

        response = openai_client.responses.create(
            conversation=conversation.id,
            input=user_text,
            extra_body={"agent_reference": {"name": agent.name, "type": "agent_reference"}},
        )
        print(f"🤖 Nutrition response status: {response.status}")
        return conversation, response
    except Exception as e:
        print(f"❌ Error analyzing nutrition data: {e}")
        return None, None

nutrition_conversation, nutrition_response = None, None
if health_agent:
    nutrition_conversation, nutrition_response = analyze_nutrition_data(health_agent)

## 5. Viewing Results & Visualizations 📊
The agent may produce text insights, disclaimers, and even chart files. Code Interpreter returns generated files as **`container_file_citation`** annotations on the response, so we read those and download each file (e.g. the nutrition chart PNG).


In [ ]:
def view_agent_response(title, response):
    """Print the agent's text answer and download any generated chart files."""
    if response is None:
        return
    print(f"\n=== {title} (status: {response.status}) ===")
    print("Response:", response.output_text, "\n")

    # Code Interpreter returns generated files as container_file_citation annotations.
    for item in (response.output or []):
        if getattr(item, "type", "") != "message":
            continue
        for content in (getattr(item, "content", None) or []):
            for ann in (getattr(content, "annotations", None) or []):
                if getattr(ann, "type", "") == "container_file_citation":
                    outname = os.path.basename(ann.filename) or f"chart_{ann.file_id}.png"
                    file_content = openai_client.containers.files.content.retrieve(
                        file_id=ann.file_id,
                        container_id=ann.container_id,
                    )
                    with open(outname, "wb") as f:
                        f.write(file_content.read())
                    print(f"🖼️ Saved generated file: {outname}")

# Display BMI calculations
if bmi_response:
    view_agent_response("BMI Calculation Results", bmi_response)

# Display nutrition analyses (this is where the chart PNG is downloaded)
if nutrition_response:
    view_agent_response("Nutrition Analysis Results", nutrition_response)

## 6. Cleanup & Best Practices
We can remove our agent and sample data if desired. In production, you might keep them for repeated usage.

### Best Practices in a Nutshell
1. **Data Handling** – Validate input data, handle missing values, properly manage file attachments.
2. **Calculations** – Provide formula steps, disclaimers, limit scope to general wellness, remind user you're not a doctor.
3. **Visualizations** – Use clear labeling and disclaimers that charts are for educational demonstrations.
4. **Security** – Monitor usage, limit access to code interpreter if dealing with proprietary data.


### 👀 See your agent live in the Foundry portal

Before deleting the agent, confirm it now exists as a first-class resource in your Foundry project:

1. In a browser, open the [Microsoft Foundry portal](https://ai.azure.com) and select your project.
2. In the top navigation select **Build**, then select **Agents** in the left pane.
3. Locate **`health-calculator-agent`** in the list — this is the very agent you just created from code. Open it to inspect its instructions and tools.
4. Return to this notebook and run the next cell to delete the agent and free up resources.

> **Note:** Because several notebooks each create an agent, your project can accumulate them quickly — so every notebook deletes its own agent at the end.

In [ ]:
def cleanup_all():
    try:
        # Delete the uploaded CSV file from the service
        if 'uploaded_file' in globals() and uploaded_file:
            openai_client.files.delete(uploaded_file.id)
            print("🗑️ Deleted uploaded file from the service.")

        # Delete the agent version if we created one
        if 'health_agent' in globals() and health_agent:
            project_client.agents.delete_version(
                agent_name=health_agent.name,
                agent_version=health_agent.version,
            )
            print("🗑️ Deleted health calculator agent.")

        # Delete local CSV file
        if 'sample_file' in globals() and sample_file and os.path.exists(sample_file):
            os.remove(sample_file)
            print("🗑️ Deleted local sample CSV file.")

    except Exception as e:
        print(f"❌ Error during cleanup: {e}")

cleanup_all()

# Congratulations! 🎉
You now have a **Health Calculator Agent** with the **Code Interpreter** tool that can:
- Perform **BMI calculations** and disclaim that it's not a doctor.
- **Analyze** simple CSV-based nutrition data and produce insights + charts.
- Return images (charts) and text-based insights.

#### Let's proceed to [3-file-search.ipynb](3-file-search.ipynb)

Happy (healthy) coding! 💪
